# §4 Individual with ML#2 filter (v12 full pool subsample 2k, V4 filter) — net of costs

Per-combo metrics and equity/drawdown curves after applying the
ML#2 booster + pooled-R:R isotonic calibrator filter.

**Cost model:** every trade is charged `contracts × $5.00` round-trip (≈ $3 retail commission + 2 ticks/side slippage on MNQ at $0.50/tick).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation._top_perf_common import (
    STARTING_EQUITY, POLICIES,
    apply_sizing, metrics_from_pnl, monte_carlo,
    load_setup,
    plot_indiv_equity, plot_indiv_dd,
    plot_combined_equity, plot_combined_dd,
    plot_ml2_indiv_equity, plot_ml2_indiv_dd,
    plot_ml2_combined_equity, plot_ml2_combined_dd,
    plot_mc_sims, plot_mc_pnl, plot_mc_sharpe, plot_mc_dd,
)

_ctx = load_setup(cost_per_contract_rt=5.0, top_strategies_path=REPO / 'evaluation' / 'top_strategies_v12_full_pool_2k.json', version='v4')
bars            = _ctx['bars']
YEARS_SPAN      = _ctx['years_span']
strategies      = _ctx['strategies']
results_raw     = _ctx['results_raw']
combined_raw    = _ctx['combined_raw']
combos_ml2      = _ctx['combos_ml2']
s4_pnl_by_combo = _ctx['s4_pnl_by_combo']
ml2_portfolio   = _ctx['ml2_portfolio']


Top-K source: top_strategies_v12_full_pool_2k.json


Test partition: 514,563 bars  2024-10-22 05:08:00 -> 2026-04-08 20:20:00
Years span: 1.461  (used to annualize Sharpe)
Applying friction: $5.00/contract RT (commission + slippage).
Loaded 500 strategies.


Loaded results_raw from cache (500 combos).


Combined unfiltered trades: 1,837,764
Loaded combos_ml2 from cache (500 combos).


ML2 portfolio trade counts: {'fixed_dollars_500': 24542}


In [2]:
rows = []
for cid, entry in s4_pnl_by_combo.items():
    pnl_base = entry['pnl_base']; risk_base = entry['risk_base']
    if len(pnl_base) == 0:
        for policy in POLICIES:
            rows.append({'combo_id': cid, 'policy': policy,
                         **metrics_from_pnl(np.array([]), YEARS_SPAN, policy=policy)})
        continue
    r_mult = np.where(risk_base > 0, pnl_base / risk_base, 0.0)
    for policy in POLICIES:
        pnl = entry['by_policy'][policy]
        rows.append({'combo_id': cid, 'policy': policy,
                     **metrics_from_pnl(pnl, YEARS_SPAN, policy=policy, r=r_mult)})
perf4 = pd.DataFrame(rows)
perf4

,combo_id,policy,n_trades,trades_per_year,win_rate,total_pnl_dollars,total_return_pct,sharpe_ratio,max_drawdown_pct,max_drawdown_dollars
0,v11_25,fixed_dollars_500,4,2.7,0.0000,-1595.49,-3.19,-2.7207,3.19,1595.49
1,v11_68,fixed_dollars_500,33,22.6,0.4242,1113.28,2.23,0.2484,5.36,2808.35
2,v11_147,fixed_dollars_500,24,16.4,0.3333,2375.93,4.75,0.6562,2.55,1337.69
3,v11_234,fixed_dollars_500,0,0.0,0.0000,0.00,0.00,0.0000,0.00,0.00
4,v11_240,fixed_dollars_500,2,1.4,0.5000,1028.08,2.06,0.3987,1.07,552.50
...,...,...,...,...,...,...,...,...,...,...
495,v11_29670,fixed_dollars_500,13,8.9,0.3077,3324.06,6.65,0.8496,5.26,2627.73
496,v11_29860,fixed_dollars_500,25,17.1,0.5200,3181.11,6.36,1.0344,3.48,1756.84
497,v11_29854,fixed_dollars_500,17,11.6,0.3529,7615.06,15.23,1.0900,4.45,2226.00
498,v11_29982,fixed_dollars_500,6,4.1,0.0000,-926.25,-1.85,-1.6090,1.85,926.25


In [3]:
plot_ml2_indiv_equity(s4_pnl_by_combo, bars, 'fixed_dollars_500')

/root/intra/scripts/evaluation/_top_perf_common.py:545: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout(); plt.show()


In [4]:
plot_ml2_indiv_dd(s4_pnl_by_combo, bars, 'fixed_dollars_500')

/root/intra/scripts/evaluation/_top_perf_common.py:564: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout(); plt.show()
